<a href="https://colab.research.google.com/github/rc2308/churn-ensemble-engine/blob/main/notebooks/02_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 — Feature Engineering

**Goal:** Apply feature engineering, encode categoricals, and create the
train/test split. Produces model-ready datasets.

**Uses:** `src/data_prep.py` + `src/features.py`


In [1]:
# Remove old copy (if any) and clone fresh to get latest src/
import os
os.chdir("/content")
!rm -rf churn-ensemble-engine
!git clone https://github.com/rc2308/churn-ensemble-engine.git
%cd churn-ensemble-engine

# Install dependencies
!pip install -q -r requirements.txt


Cloning into 'churn-ensemble-engine'...
remote: Enumerating objects: 37, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 37 (delta 14), reused 20 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (37/37), 630.05 KiB | 2.41 MiB/s, done.
Resolving deltas: 100% (14/14), done.
/content/churn-ensemble-engine
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 76.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 92.7 MB/s eta 0:00:00


In [2]:
from google.colab import files
uploaded = files.upload()   # select BankChurners.csv


Saving BankChurners.csv to BankChurners.csv


In [3]:
import pandas as pd
from src.data_prep import load_data, clean_data, split_data

# Load and clean
df = load_data("BankChurners.csv")
df = clean_data(df)

print("Shape after cleaning:", df.shape)
df.head()


Shape after cleaning: (10127, 20)


,Customer_Age,Gender,Dependent_count,Education_Level,Marital_Status,Income_Category,Card_Category,Months_on_book,Total_Relationship_Count,Months_Inactive_12_mon,Contacts_Count_12_mon,Credit_Limit,Total_Revolving_Bal,Avg_Open_To_Buy,Total_Amt_Chng_Q4_Q1,Total_Trans_Amt,Total_Trans_Ct,Total_Ct_Chng_Q4_Q1,Avg_Utilization_Ratio,Churn
0,45,M,3,High School,Married,$60K - $80K,Blue,39,5,1,3,12691.0,777,11914.0,1.335,1144,42,1.625,0.061,0
1,49,F,5,Graduate,Single,Less than $40K,Blue,44,6,1,2,8256.0,864,7392.0,1.541,1291,33,3.714,0.105,0
2,51,M,3,Graduate,Married,$80K - $120K,Blue,36,4,1,0,3418.0,0,3418.0,2.594,1887,20,2.333,0.000,0
3,40,F,4,High School,Unknown,Less than $40K,Blue,34,3,4,1,3313.0,2517,796.0,1.405,1171,20,2.333,0.760,0
4,40,M,3,Uneducated,Married,$60K - $80K,Blue,21,5,1,0,4716.0,0,4716.0,2.175,816,28,2.500,0.000,0


In [4]:
from src.features import engineer_features, encode_categoricals

# Step 1: create new ratio features
df = engineer_features(df)
print("New features added:")
print(df[["Amt_per_Transaction", "Contacts_per_Month"]].head())


New features added:
   Amt_per_Transaction  Contacts_per_Month
0            26.604651            0.075000
1            37.970588            0.044444
2            89.857143            0.000000
3            55.761905            0.028571
4            28.137931            0.000000


In [5]:
# Step 2: one-hot encode text columns
df_encoded = encode_categoricals(df)

print("Shape before encoding:", df.shape)
print("Shape after encoding:", df_encoded.shape)
print("\nColumns now:", df_encoded.columns.tolist())
df_encoded.head()


Shape before encoding: (10127, 22)
Shape after encoding: (10127, 35)

Columns now: ['Customer_Age', 'Dependent_count', 'Months_on_book', 'Total_Relationship_Count', 'Months_Inactive_12_mon', 'Contacts_Count_12_mon', 'Credit_Limit', 'Total_Revolving_Bal', 'Avg_Open_To_Buy', 'Total_Amt_Chng_Q4_Q1', 'Total_Trans_Amt', 'Total_Trans_Ct', 'Total_Ct_Chng_Q4_Q1', 'Avg_Utilization_Ratio', 'Churn', 'Amt_per_Transaction', 'Contacts_per_Month', 'Gender_M', 'Education_Level_Doctorate', 'Education_Level_Graduate', 'Education_Level_High School', 'Education_Level_Post-Graduate', 'Education_Level_Uneducated', 'Education_Level_Unknown', 'Marital_Status_Married', 'Marital_Status_Single', 'Marital_Status_Unknown', 'Income_Category_$40K - $60K', 'Income_Category_$60K - $80K', 'Income_Category_$80K - $120K', 'Income_Category_Less than $40K', 'Income_Category_Unknown', 'Card_Category_Gold', 'Card_Category_Platinum', 'Card_Category_Silver']


,Customer_Age,Dependent_count,Months_on_book,Total_Relationship_Count,Months_Inactive_12_mon,Contacts_Count_12_mon,Credit_Limit,Total_Revolving_Bal,Avg_Open_To_Buy,Total_Amt_Chng_Q4_Q1,...,Marital_Status_Single,Marital_Status_Unknown,Income_Category_$40K - $60K,Income_Category_$60K - $80K,Income_Category_$80K - $120K,Income_Category_Less than $40K,Income_Category_Unknown,Card_Category_Gold,Card_Category_Platinum,Card_Category_Silver
0,45,3,39,5,1,3,12691.0,777,11914.0,1.335,...,False,False,False,True,False,False,False,False,False,False
1,49,5,44,6,1,2,8256.0,864,7392.0,1.541,...,True,False,False,False,False,True,False,False,False,False
2,51,3,36,4,1,0,3418.0,0,3418.0,2.594,...,False,False,False,False,True,False,False,False,False,False
3,40,4,34,3,4,1,3313.0,2517,796.0,1.405,...,False,True,False,False,False,True,False,False,False,False
4,40,3,21,5,1,0,4716.0,0,4716.0,2.175,...,False,False,False,True,False,False,False,False,False,False


In [6]:
# Step 3: stratified 80/20 split
X_train, X_test, y_train, y_test = split_data(df_encoded, target_col="Churn")

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("Train churn rate:", y_train.mean().round(3))
print("Test churn rate:", y_test.mean().round(3))


X_train shape: (8101, 34)
X_test shape: (2026, 34)
Train churn rate: 0.161
Test churn rate: 0.16


In [7]:
# Save the splits so other notebooks can reuse them
import os
os.makedirs("data/processed", exist_ok=True)

X_train.to_csv("data/processed/X_train.csv", index=False)
X_test.to_csv("data/processed/X_test.csv", index=False)
y_train.to_csv("data/processed/y_train.csv", index=False)
y_test.to_csv("data/processed/y_test.csv", index=False)

# Also save the full encoded dataset (for production retraining + clustering)
df_encoded.to_csv("data/processed/full_encoded.csv", index=False)

print("Processed data saved to data/processed/")


Processed data saved to data/processed/


## Feature Engineering Summary

- **Engineered features:** `Amt_per_Transaction`, `Contacts_per_Month`
- **Encoding:** One-hot encoded all categorical columns (drop_first=True)
- **Split:** Stratified 80/20 — both sets preserve ~16% churn rate
- **Saved:** X_train, X_test, y_train, y_test, and full_encoded to `data/processed/`

These datasets feed into clustering (Stage 1) and modeling (Stage 2).
